In [58]:
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
from sklearn.tree import export_graphviz

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_squared_error as MSE

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.metrics import ndcg_score

import torch
import torch.nn as nn
import torch.optim as optim

In [39]:
df = pd.read_csv('final_df.csv')
df = df.drop(columns=df.columns[0])
df = df.drop('dew_point_temperature', axis=1)
df

,id,day,month,year,local_time,air_temperature,atm_pressure_station_level,atm_pressure_sea_level,atm_pressure_3hour_change,humidity,...,"clouds_low_level_Кучево-дождевые волокнистые (часто с наковальней), либо с кучево-дождевыми лысыми, кучевыми, слоистыми, разорванно-дождевыми, либо без них.","clouds_low_level_Кучево-дождевые лысые с кучевыми, слоисто-кучевыми или слоистыми, либо без них.","clouds_low_level_Кучевые и слоисто-кучевые (но не слоисто-кучевые, образовавшиеся из кучевых), основания расположены на разных уровнях.","clouds_low_level_Кучевые плоские или кучевые разорванные, или те и другие вместе, не относящиеся к облакам плохой погоды.","clouds_low_level_Кучевые средние или мощные или вместе с кучевыми разорванными, или с кучевыми плоскими, или со слоисто-кучевыми, либо без них; основания всех этих облаков расположены на одном уровне.","clouds_low_level_Слоисто-кучевые, образовавшиеся из кучевых.","clouds_low_level_Слоисто-кучевые, образовавшиеся не из кучевых.","clouds_low_level_Слоисто-кучевых, слоистых, кучевых или кучево-дождевых облаков нет.","clouds_low_level_Слоистые разорванные или кучевые разорванные облака плохой погоды, либо те и другие вместе (разорванно-дождевые); обычно расположены под слоистыми или слоисто-дождевыми облаками.","clouds_low_level_Слоистые туманообразные или слоистые разорванные, либо те и другие, но не относящиеся к облакам плохой погоды."
0,0,26,2,2024,21,4.6,742.1,763.4,1.1,98.0,...,False,False,False,False,False,False,True,False,False,False
1,1,26,2,2024,18,4.8,741.0,762.2,0.9,99.0,...,False,False,False,False,False,False,True,False,False,False
2,2,26,2,2024,15,4.6,740.1,761.3,0.3,99.0,...,False,False,False,False,False,False,False,False,False,True
3,3,26,2,2024,12,4.4,739.8,761.0,0.4,100.0,...,False,False,False,False,False,False,True,False,False,False
4,4,26,2,2024,9,4.8,739.4,760.6,0.2,100.0,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55529,55529,1,2,2005,15,-5.5,732.5,753.6,1.5,90.0,...,False,False,False,False,False,True,False,False,False,False
55530,55530,1,2,2005,12,-7.1,731.0,752.2,1.7,88.0,...,False,False,False,False,False,True,False,False,False,False
55531,55531,1,2,2005,9,-8.3,729.3,750.5,1.1,92.0,...,False,False,False,False,False,True,False,False,False,False
55532,55532,1,2,2005,6,-8.5,728.2,749.4,0.3,89.0,...,False,False,False,False,False,True,False,False,False,False


Разделяем датасет на тренировочный, валидационный и проверочный:
- train: 01.02.2005 - 13.06.2018   ~70%
- val: 14.06.2018 – 20.04.2021    ~15%
- test: 21.04.2021 – 26.02.2024   ~15% 

In [40]:
# Создаем фильтры для каждого набора данных
train_filter = (df['year'] < 2018) | ((df['year'] == 2018) & (df['month'] < 6)) | ((df['year'] == 2018) & (df['month'] == 6) & (df['day'] <= 13))
val_filter = ((df['year'] == 2018) & (df['month'] == 6) & (df['day'] >= 14)) | ((df['year'] > 2018) & (df['year'] < 2021)) | ((df['year'] == 2021) & (df['month'] < 4)) | ((df['year'] == 2021) & (df['month'] == 4) & (df['day'] <= 20))
test_filter = ((df['year'] == 2021) & (df['month'] == 4) & (df['day'] >= 21)) | (df['year'] > 2021) | ((df['year'] == 2021) & (df['month'] > 4))

# Фильтруем DataFrame для каждого набора данных
train_df = df[train_filter]
val_df = df[val_filter]
test_df = df[test_filter]

# Выводим полученные DataFrame
print("Train set:")
print(train_df)
print("\nValidation set:")
print(val_df)
print("\nTest set:")
print(test_df)

Train set:
          id  day  month  year  local_time  air_temperature  \
16662  16662   13      6  2018          21             21.1   
16663  16663   13      6  2018          18             23.0   
16664  16664   13      6  2018          15             23.8   
16665  16665   13      6  2018          12             20.7   
16666  16666   13      6  2018           9             17.3   
...      ...  ...    ...   ...         ...              ...   
55529  55529    1      2  2005          15             -5.5   
55530  55530    1      2  2005          12             -7.1   
55531  55531    1      2  2005           9             -8.3   
55532  55532    1      2  2005           6             -8.5   
55533  55533    1      2  2005           3             -8.3   

       atm_pressure_station_level  atm_pressure_sea_level  \
16662                       736.3                   755.7   
16663                       736.0                   755.2   
16664                       736.0                

In [41]:
train_df_wo_temp = train_df.drop(columns=['air_temperature'])

val_df_wo_temp = val_df.drop(columns=['air_temperature'])

test_df_wo_temp = test_df.drop(columns=['air_temperature'])

In [32]:
x = train_df_wo_temp
y = train_df[['air_temperature']]

tree_counts = [1, 100, 1000]

mae_stat = []
mape_stat = []

for count in tree_counts:
    # print(count)
    forest = RandomForestRegressor(n_estimators=count, random_state=10)
    forest.fit(x, y)

    predictions = forest.predict(test_df_wo_temp)

    current_mae = MAE(test_df['air_temperature'], predictions)
    current_mape = MAPE(test_df['air_temperature'], predictions)

    mae_stat.append(current_mae)
    mape_stat.append(current_mape)

    print(current_mae)
    print(current_mape)

c:\Users\korde\anaconda3\Lib\site-packages\sklearn\base.py:1151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.19333813467770977
1621749955841.0745


c:\Users\korde\anaconda3\Lib\site-packages\sklearn\base.py:1151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.09574792942023773
973590556823.2505


c:\Users\korde\anaconda3\Lib\site-packages\sklearn\base.py:1151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


KeyboardInterrupt: 

Бустинги

In [59]:
x = train_df_wo_temp
y = train_df[['air_temperature']]

tree_counts = [1, 5, 10, 50, 100, 200, 1000, 10000]
random_state = list(range(1, 1001))
current_mae = 5000
current_mape = 5000
print(current_mae)
print(current_mape)

for count in tree_counts:
    for current_random_state in random_state:
        final_count = count
        final_random_state = current_random_state
        tree = DecisionTreeRegressor(max_depth=count, random_state=current_random_state)
        tree.fit(x, y)

        predictions = tree.predict(val_df_wo_temp)

        mae = MAE(val_df['air_temperature'], predictions)
        mape = MAPE(val_df['air_temperature'], predictions)
        print(mae)

        if mae < current_mae and mape < current_mape:

            current_mae = mae
            current_mape = mape

            print("current random state: ", current_random_state)
            print("max depth: ", count)
            print("MAE: ", current_mae)
            print("MAPE: ", current_mape)
            print("==============================================")
        
        else:
            print("current random state: ", current_random_state)
            print("max depth: ", count)
            print("MAE and MAPE stoped decreasing")
            print("changing a number of trees")
            print("----------------------------------------------")
            break

test_prediction = tree.predict(test_df_wo_temp)
mae = MAE(test_df['air_temperature'], test_prediction)
mape = MAPE(test_df['air_temperature'], test_prediction)
mse = MSE(test_df['air_temperature'], test_prediction)
print("final random state: ", final_random_state)
print("final max depth: ", final_count)
print("test MAE: ", mae)
print("test RMSE: ", np.sqrt(mse))
print("test MAPE: ", mape)

5000
5000
6.054574688282845
current random state:  1
max depth:  1
MAE and MAPE stoped decreasing
changing a number of trees
----------------------------------------------
3.4781673716554273
current random state:  1
max depth:  5
MAE and MAPE stoped decreasing
changing a number of trees
----------------------------------------------
3.417644310902194
current random state:  1
max depth:  10
MAE and MAPE stoped decreasing
changing a number of trees
----------------------------------------------
4.080711473975798
current random state:  1
max depth:  50
MAE and MAPE stoped decreasing
changing a number of trees
----------------------------------------------
4.080711473975798
current random state:  1
max depth:  100
MAE and MAPE stoped decreasing
changing a number of trees
----------------------------------------------
4.080711473975798
current random state:  1
max depth:  200
MAE and MAPE stoped decreasing
changing a number of trees
----------------------------------------------
4.080711473

In [32]:
test_prediction_df = pd.DataFrame({
    'id': test_df['id'],
    'test_prediction': test_prediction,
    'air_temperature': test_df['air_temperature']
})

test_prediction_df

,id,test_prediction,air_temperature
0,0,4.6,4.6
1,1,4.7,4.8
2,2,4.7,4.6
3,3,4.6,4.4
4,4,5.0,4.8
...,...,...,...
8326,8326,10.4,10.3
8327,8327,6.6,6.9
8328,8328,-1.8,-1.7
8329,8329,-0.7,-0.7


In [46]:
# намеренно увеличиваю на 30 значения предсказания для наглядного запараллеливания графиков
test_prediction_df['test_prediction'] += 30

plt.figure(figsize=(180, 40))  # установка ширины и высоты графика

# Отрисовка графика
plt.plot(test_prediction_df['id'], test_prediction_df['air_temperature'], label='air_temperature', color='red')
plt.plot(test_prediction_df['id'], test_prediction_df['test_prediction'], label='test_prediction', color='blue')

# Добавляем подписи к осям и легенду
plt.xlabel('id')
plt.ylabel('Value')
plt.legend()

# Показываем график
plt.show()

NameError: name 'test_prediction_df' is not defined

lightGBM не принимает названия с русскими буквами - нужно поубирать всё

In [46]:
all_columns = df.columns.tolist()
all_columns

['id',
 'day',
 'month',
 'year',
 'local_time',
 'air_temperature',
 'atm_pressure_station_level',
 'atm_pressure_sea_level',
 'atm_pressure_3hour_change',
 'humidity',
 'wind_speed',
 'cloudiness',
 'cloud_cover',
 'clouds_low_level_height',
 'horizontal_visibility_range_km',
 'dew_point_temperature',
 'precipitation_amount_mm',
 'precipitation_accumulation_time_hrs',
 'soil_surface_condition_wo_snow_or_ice_cover',
 'soil_surface_condition_with_snow_or_ice_cover',
 'snow_depth_cm',
 'wind_direction_Ветер, дующий с востока',
 'wind_direction_Ветер, дующий с востоко-северо-востока',
 'wind_direction_Ветер, дующий с востоко-юго-востока',
 'wind_direction_Ветер, дующий с запада',
 'wind_direction_Ветер, дующий с западо-северо-запада',
 'wind_direction_Ветер, дующий с западо-юго-запада',
 'wind_direction_Ветер, дующий с севера',
 'wind_direction_Ветер, дующий с северо-востока',
 'wind_direction_Ветер, дующий с северо-запада',
 'wind_direction_Ветер, дующий с северо-северо-востока',
 'wind

In [15]:
# Создаем словарь для перевода названий столбцов
translation_dict = {
    'id': 'id',
    'day': 'day',
    'month': 'month',
    'year': 'year',
    'local_time': 'local_time',
    'air_temperature': 'air_temperature',
    'atm_pressure_station_level': 'atm_pressure_station_level',
    'atm_pressure_sea_level': 'atm_pressure_sea_level',
    'atm_pressure_3hour_change': 'atm_pressure_3hour_change',
    'humidity': 'humidity',
    'wind_speed': 'wind_speed',
    'cloudiness': 'cloudiness',
    'cloud_cover': 'cloud_cover',
    'clouds_low_level_height': 'clouds_low_level_height',
    'horizontal_visibility_range_km': 'horizontal_visibility_range_km',
    'dew_point_temperature': 'dew_point_temperature',
    'precipitation_amount_mm': 'precipitation_amount_mm',
    'precipitation_accumulation_time_hrs': 'precipitation_accumulation_time_hrs',
    'soil_surface_condition_wo_snow_or_ice_cover': 'soil_surface_condition_wo_snow_or_ice_cover',
    'soil_surface_condition_with_snow_or_ice_cover': 'soil_surface_condition_with_snow_or_ice_cover',
    'snow_depth_cm': 'snow_depth_cm',
    'wind_direction_Ветер, дующий с востока': 'wind_direction_east',
    'wind_direction_Ветер, дующий с востоко-северо-востока': 'wind_direction_east-north-east',
    'wind_direction_Ветер, дующий с востоко-юго-востока': 'wind_direction_east-south-east',
    'wind_direction_Ветер, дующий с запада': 'wind_direction_west',
    'wind_direction_Ветер, дующий с западо-северо-запада': 'wind_direction_west-north-west',
    'wind_direction_Ветер, дующий с западо-юго-запада': 'wind_direction_west-south-west',
    'wind_direction_Ветер, дующий с севера': 'wind_direction_north',
    'wind_direction_Ветер, дующий с северо-востока': 'wind_direction_north-east',
    'wind_direction_Ветер, дующий с северо-запада': 'wind_direction_north-west',
    'wind_direction_Ветер, дующий с северо-северо-востока': 'wind_direction_north-north-east',
    'wind_direction_Ветер, дующий с северо-северо-запада': 'wind_direction_north-north-west',
    'wind_direction_Ветер, дующий с юга': 'wind_direction_south',
    'wind_direction_Ветер, дующий с юго-востока': 'wind_direction_south-east',
    'wind_direction_Ветер, дующий с юго-запада': 'wind_direction_south-west',
    'wind_direction_Ветер, дующий с юго-юго-востока': 'wind_direction_south-south-east',
    'wind_direction_Ветер, дующий с юго-юго-запада': 'wind_direction_south-south-west',
    'wind_direction_Штиль, безветрие': 'wind_direction_calm',
    'current_weather_  Диаметр изморозевого отложения составляет 1 мм.': 'current_weather_frost-deposit-diameter-1mm',
    'current_weather_  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_frost-deposit-diameter-2mm',
    'current_weather_  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_frost-deposit-diameter-3mm',
    'current_weather_  Диаметр изморозевого отложения составляет 4 мм.': 'current_weather_frost-deposit-diameter-4mm',
    'current_weather_  Диаметр отложения мокрого снега составляет 5 мм.': 'current_weather_wet-snow-deposit-diameter-5mm',
    'current_weather_  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_ice-deposit-diameter-1mm',
    'current_weather_  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_ice-snow-deposit-diameter-2mm',
    'current_weather_  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_ice-snow-deposit-diameter-4mm',
    'current_weather_  Диаметр смешанного отложения составляет 2 мм.': 'current_weather_mixed-sediment-diameter-2mm',
    'current_weather_  Диаметр смешанного отложения составляет 3 мм.': 'current_weather_mixed-sediment-diameter-3mm',
    'current_weather_Алмазная пыль (с туманом или без него). ': 'current_weather_Diamond dust with or without fog',
    'current_weather_Алмазная пыль (с туманом или без него).  Диаметр изморозевого отложения составляет 1 мм.': 'current_weather_Diamond dust with or without fog Frost deposit diameter is 1 mm',
    'current_weather_Более или менее сплошной слой приземного или ледяного тумана на станции, на море или на суше, высотой не более 2 м над сушей или не более 10 м над морем. ': 'current_weather_More or less continuous layer of ground or ice fog at the station at sea or on land not exceeding 2 m above ground or 10 m above sea level',
    'current_weather_Видимость ухудшена дымом, например: от степных или лесных пожаров, промышленных предприятий или вулканическим пеплом. ': 'current_weather_Visibility reduced by smoke from steppe or forest fires industrial plants or volcanic ash',
    'current_weather_Видна молния, грома не слышно. ': 'current_weather_Lightning visible thunder not audible',
    'current_weather_Гроза (с осадками или без них). ': 'current_weather_Thunderstorm with or without precipitation',
    'current_weather_Гроза (с осадками или без них).  Максимальный диаметр градин составляет 10 мм.': 'current_weather_Thunderstorm with or without precipitation Maximum hailstone diameter is 10 mm',
    'current_weather_Гроза (с осадками или без них).  Максимальный диаметр градин составляет 4 мм.': 'current_weather_Thunderstorm with or without precipitation Maximum hailstone diameter is 4 mm',
    'current_weather_Гроза вместе с пыльной или песчаной бурей в срок наблюдения. ': 'current_weather_Thunderstorm accompanied by dust or sandstorm within observation period',
    'current_weather_Гроза сильная без града, но с дождем и/или снегом в срок наблюдения. ': 'current_weather_Strong thunderstorm without hail but with rain and/or snow within observation period',
    'current_weather_Гроза слабая или умеренная без града, но с дождем и/или снегом в срок наблюдения. ': 'current_weather_Light or moderate thunderstorm without hail but with rain and/or snow within observation period',
    'current_weather_Гроза слабая или умеренная без града, но с дождем и/или снегом в срок наблюдения.  Максимальный диаметр градин составляет 1 мм.': 'current_weather_Light or moderate thunderstorm without hail but with rain and/or snow within observation period Maximum hailstone diameter is 1 mm',
    'current_weather_Гроза слабая или умеренная с градом в срок наблюдения. ': 'current_weather_Light or moderate thunderstorm with hail within observation period',
    'current_weather_Гроза, но без осадков, в срок наблюдения. ': 'current_weather_Thunderstorm, but without precipitation within observation period',
    'current_weather_Дождь (незамерзающий) неливневый. ': 'current_weather_Non-freezing non-showery rain',
    'current_weather_Дождь (незамерзающий) неливневый.  Диаметр отложения мокрого снега составляет 10 мм.': 'current_weather_Non-freezing non-showery rain Wet snow deposit diameter is 10 mm',
    'current_weather_Дождь замерзающий слабый. ': 'current_weather_Weak freezing rain',
    'current_weather_Дождь замерзающий слабый.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Weak freezing rain Ice deposit diameter is 1 mm',
    'current_weather_Дождь замерзающий слабый.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Weak freezing rain Ice deposit diameter is 2 mm',
    'current_weather_Дождь или морось со снегом слабые. ': 'current_weather_Weak rain or drizzle with snow',
    'current_weather_Дождь или морось со снегом слабые.  Диаметр отложения мокрого снега составляет 14 мм.': 'current_weather_Weak rain or drizzle with snow Wet snow deposit diameter is 14 mm',
    'current_weather_Дождь или морось со снегом умеренные или сильные. ': 'current_weather_Moderate or heavy rain or drizzle with snow',
    'current_weather_Дождь незамерзающий непрерывный сильный в срок наблюдения. ': 'current_weather_Continuous heavy non-freezing rain within observation period',
    'current_weather_Дождь незамерзающий непрерывный слабый в срок наблюдения. ': 'current_weather_Continuous weak non-freezing rain within observation period',
    'current_weather_Дождь незамерзающий непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 12 мм.': 'current_weather_Continuous weak non-freezing rain within observation period Wet snow deposit diameter is 12 mm',
    'current_weather_Дождь незамерзающий непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Continuous weak non-freezing rain within observation period Ice deposit diameter is 2 mm',
    'current_weather_Дождь незамерзающий непрерывный слабый в срок наблюдения.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Continuous weak non-freezing rain within observation period Ice deposit diameter is 4 mm',
    'current_weather_Дождь незамерзающий непрерывный слабый в срок наблюдения.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Moderate continuous non-freezing rain within observation period',
    'current_weather_Дождь незамерзающий непрерывный умеренный в срок наблюдения. ': 'current_weather_Intermittent heavy non-freezing rain within observation period',
    'current_weather_Дождь незамерзающий с перерывами сильный в срок наблюдения. ': 'current_weather_Intermittent weak non-freezing rain within observation period',
    'current_weather_Дождь незамерзающий с перерывами слабый в срок наблюдения. ': 'current_weather_Intermittent moderate non-freezing rain within observation period',
    'current_weather_Дождь незамерзающий с перерывами умеренный в срок наблюдения. ': 'current_weather_Non-showery rain with snow or ice pellets',
    'current_weather_Дождь со снегом или ледяная крупа неливневые. ': 'current_weather_Rain with snow or ice pellets non-showery',
    'current_weather_Дымка. ': 'current_weather_Mist',
    'current_weather_Дымка.  Диаметр изморозевого отложения составляет 1 мм.': 'current_weather_Mist Diameter of freezing deposit is 1 mm',
    'current_weather_Дымка.  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_Mist Diameter of freezing deposit is 2 mm',
    'current_weather_Дымка.  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_Mist Diameter of freezing deposit is 3 mm',
    'current_weather_Дымка.  Диаметр изморозевого отложения составляет 8 мм.': 'current_weather_Mist Diameter of freezing deposit is 8 mm',
    'current_weather_Дымка.  Диаметр изморозевого отложения составляет 9 мм.': 'current_weather_Mist Diameter of freezing deposit is 9 mm',
    'current_weather_Дымка.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Mist Diameter of wet snow deposit is 2 mm',
    'current_weather_Дымка.  Диаметр отложения мокрого снега составляет 4 мм.': 'current_weather_Mist Diameter of wet snow deposit is 4 mm',
    'current_weather_Дымка.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Mist Diameter of ice deposit is 1 mm',
    'current_weather_Дымка.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Mist Diameter of ice deposit is 2 mm',
    'current_weather_Дымка.  Диаметр отложения при гололеде составляет 3 мм.': 'current_weather_Mist Diameter of ice deposit is 3 mm',
    'current_weather_Дымка.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Mist Diameter of ice deposit is 4 mm',
    'current_weather_Дымка.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Mist Diameter of mixed deposit is 1 mm',
    'current_weather_Замерзающая морось или замерзающий дождь неливневые. ': 'current_weather_Freezing drizzle or freezing rain non-showery',
    'current_weather_Замерзающая морось или замерзающий дождь неливневые.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Freezing drizzle or freezing rain non-showery Diameter of ice deposit is 1 mm',
    'current_weather_Замерзающая морось или замерзающий дождь неливневые.  Диаметр отложения при гололеде составляет 3 мм.': 'current_weather_Freezing drizzle or freezing rain non-showery Diameter of ice deposit is 3 mm',
    'current_weather_Клочья приземного или ледяного тумана на станции, на море или на суше, высотой не более 2 м над сушей или не более 10 м над морем. ': 'current_weather_Clumps of ground or ice fog at the station at sea or on land not higher than 2 m above the ground or not more than 10 m above sea level',
    'current_weather_Ледяная крупа.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Small hail Diameter of ice deposit is 2 mm',
    'current_weather_Ливневая снежная крупа или небольшой град с дождем или без него, или дождь со снегом слабые в срок наблюдения или за последний час. ': 'current_weather_Showers of snow pellets or small hail with rain or without it or weak rain with snow within the observation period or in the last hour',
    'current_weather_Ливневый град с дождем или без него, или дождь со снегом без грома слабый в срок наблюдения или за последний час. ': 'current_weather_Weak hail with rain or without it or weak rain with snow without thunder within the observation period or in the last hour',
    'current_weather_Ливневый град, или дождь и град. ': 'current_weather_Thunderstorm with hail, or rain and hail',
    'current_weather_Ливневый снег или ливневый дождь и снег. ': 'current_weather_Snowstorm or snowstorm and snow',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час. ': 'current_weather_Light snowstorm within the observation period or in the last hour',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр отложения мокрого снега составляет 1 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of wet snow deposit is 1 mm',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр отложения мокрого снега составляет 5 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of wet snow deposit is 5 mm',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр отложения мокрого снега составляет 6 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of wet snow deposit is 6 mm',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of ice deposit is 2 mm',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of ice deposit is 4 mm',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of mixed deposit is 1 mm',
    'current_weather_Ливневый снег слабый в срок наблюдения или за последний час.  Диаметр смешанного отложения составляет 4 мм.': 'current_weather_Light snowstorm within the observation period or in the last hour Diameter of mixed deposit is 4 mm',
    'current_weather_Ливневый снег умеренный или сильный в срок наблюдения или за последний час. ': 'current_weather_Moderate or heavy snowstorm within the observation period or in the last hour',
    'current_weather_Ливневый снег умеренный или сильный в срок наблюдения или за последний час.  Диаметр отложения мокрого снега составляет 1 мм.': 'current_weather_Moderate or heavy snowstorm within the observation period or in the last hour Diameter of wet snow deposit is 1 mm',
    'current_weather_Ливневый снег умеренный или сильный в срок наблюдения или за последний час.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Moderate or heavy snowstorm within the observation period or in the last hour Diameter of wet snow deposit is 2 mm',
    'current_weather_Ливневый снег умеренный или сильный в срок наблюдения или за последний час.  Диаметр отложения мокрого снега составляет 5 мм.': 'current_weather_Moderate or heavy snowstorm within the observation period or in the last hour Diameter of wet snow deposit is 5 mm',
    'current_weather_Ливневый снег умеренный или сильный в срок наблюдения или за последний час.  Диаметр смешанного отложения составляет 4 мм.': 'current_weather_Moderate or heavy snowstorm within the observation period or in the last hour Diameter of mixed deposit is 4 mm',
    'current_weather_Ливневый(ые) дождь(и) очень сильный(ые) в срок наблюдения или за последний час. ': 'current_weather_Very heavy rain or rains within the observation period or in the last hour',
    'current_weather_Ливневый(ые) дождь(и) слабый(ые) в срок наблюдения или за последний час. ': 'current_weather_Light rain or rains within the observation period or in the last hour',
    'current_weather_Ливневый(ые) дождь(и) слабый(ые) в срок наблюдения или за последний час.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Light rain or rains within the observation period or in the last hour Diameter of ice deposit is 2 mm',
    'current_weather_Ливневый(ые) дождь(и) слабый(ые) в срок наблюдения или за последний час.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Light rain or rains within the observation period or in the last hour Diameter of ice deposit is 4 mm',
    'current_weather_Ливневый(ые) дождь(и) слабый(ые) в срок наблюдения или за последний час.  Максимальный диаметр градин составляет 4 мм.': 'current_weather_Light rain or rains within the observation period or in the last hour Maximum hail diameter is 4 mm',
    'current_weather_Ливневый(ые) дождь(и) со снегом слабый(ые) в срок наблюдения или за последний час. ': 'current_weather_Light rain or rains with snow within the observation period or in the last hour',
    'current_weather_Ливневый(ые) дождь(и) умеренный(ые) или сильный(ые) в срок наблюдения или за последний час. ': 'current_weather_Moderate or heavy rain or rains within the observation period or in the last hour',
    'current_weather_Ливневый(ые) дождь(и). ': 'current_weather_Rain or rains',
    'current_weather_Ливневый(ые) дождь(и).  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Rain or rains Diameter of ice deposit is 2 mm',
    'current_weather_Мгла. ': 'current_weather_Fog',
    'current_weather_Морось (незамерзающая) или снежные зерна неливневые. ': 'current_weather_Drizzle non-freezing or snow grains non-showery',
    'current_weather_Морось (незамерзающая) или снежные зерна неливневые.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Drizzle non-freezing or snow grains non-showery Diameter of ice deposit is 1 mm',
    'current_weather_Морось (незамерзающая) или снежные зерна неливневые.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Drizzle non-freezing or snow grains non-showery Diameter of ice deposit is 2 mm',
    'current_weather_Морось (незамерзающая) или снежные зерна неливневые.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Drizzle non-freezing or snow grains non-showery Diameter of ice deposit is 4 mm',
    'current_weather_Морось замерзающая слабая. ': 'current_weather_Light freezing drizzle',
    'current_weather_Морось замерзающая слабая.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Light freezing drizzle Diameter of ice deposit is 1 mm',
    'current_weather_Морось замерзающая слабая.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Light freezing drizzle Diameter of ice deposit is 2 mm',
    'current_weather_Морось замерзающая слабая.  Диаметр отложения при гололеде составляет 3 мм.': 'current_weather_Light freezing drizzle Diameter of ice deposit is 3 mm',
    'current_weather_Морось замерзающая умеренная или сильная. ': 'current_weather_Moderate or heavy freezing drizzle',
    'current_weather_Морось и дождь слабые. ': 'current_weather_Light drizzle and rain',
    'current_weather_Морось незамерзающая непрерывная слабая в срок наблюдения. ': 'current_weather_Light continuous non-freezing drizzle within the observation period',
    'current_weather_Морось незамерзающая непрерывная слабая в срок наблюдения.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Light continuous non-freezing drizzle within the observation period Diameter of ice deposit is 1 mm',
    'current_weather_Морось незамерзающая непрерывная слабая в срок наблюдения.  Диаметр отложения при гололеде составляет 3 мм.': 'current_weather_Light continuous non-freezing drizzle within the observation period Diameter of ice deposit is 3 mm',
    'current_weather_Морось незамерзающая непрерывная слабая в срок наблюдения.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Light continuous non-freezing drizzle within the observation period Diameter of ice deposit is 4 mm',
    'current_weather_Морось незамерзающая непрерывная умеренная в срок наблюдения. ': 'current_weather_Light continuous non-freezing drizzle within the observation period',
    'current_weather_Морось незамерзающая с перерывами слабая в срок наблюдения. ': 'current_weather_Light intermittent non-freezing drizzle within the observation period',
    'current_weather_Морось незамерзающая с перерывами слабая в срок наблюдения.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Light intermittent non-freezing drizzle within the observation period Diameter of wet snow deposit is 2 mm',
    'current_weather_Морось незамерзающая с перерывами умеренная в срок наблюдения. ': 'current_weather_Light intermittent non-freezing drizzle within the observation period',
    'current_weather_Облака в целом образовывались или развивались. ': 'current_weather_Moderate intermittent non-freezing drizzle within the observation period',
    'current_weather_Облака в целом рассеиваются или становятся менее развитыми. ': 'current_weather_Clouds generally dissipating or becoming less developed',
    'current_weather_Отдельные кристаллы снега в виде звездочек (с туманом или без него).  Диаметр изморозевого отложения составляет 5 мм.': 'current_weather_Individual snow crystals in the form of stars with or without fog Diameter of ice deposit is 5 mm',
    'current_weather_Сильная снежная низовая метель в целом высокая (выше уровня глаз наблюдателя). ': 'current_weather_Heavy snowstorm overall high above the observers eye level',
    'current_weather_Сильный поземок в целом низкий (ниже уровня глаз наблюдателя). ': 'current_weather_Strong fog overall low below the observers eye level',
    'current_weather_Слабая или умеренная пыльная или песчаная буря ослабела за последний час. ': 'current_weather_Weak or moderate dusty or sandy storm has weakened over the past hour',
    'current_weather_Слабая или умеренная снежная низовая метель в целом высокая (выше уровня глаз наблюдателя). ': 'current_weather_Weak or moderate snowstorm overall high above the observers eye level',
    'current_weather_Слабый дождь в срок наблюдения. Гроза в течение последнего часа, но не в срок наблюдения. ': 'current_weather_Weak rain within the observation period Thunderstorm during the last hour but not within the observation period',
    'current_weather_Слабый или умеренный поземок в целом низкий (ниже уровня глаз наблюдателя). ': 'current_weather_Weak or moderate fog overall low below the observers eye level',
    'current_weather_Слабый или умеренный поземок в целом низкий (ниже уровня глаз наблюдателя).  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Weak or moderate fog overall low below the observers eye level Diameter of wet snow deposit is 2 mm',
    'current_weather_Снег неливневый. ': 'current_weather_Non-showery snow',
    'current_weather_Снег неливневый.  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_Non-showery snow Diameter of ice deposit is 2 mm',
    'current_weather_Снег неливневый.  Диаметр изморозевого отложения составляет 4 мм.': 'current_weather_Non-showery snow Diameter of ice deposit is 4 mm',
    'current_weather_Снег неливневый.  Диаметр отложения мокрого снега составляет 3 мм.': 'current_weather_Non-showery snow Diameter of wet snow deposit is 3 mm',
    'current_weather_Снег неливневый.  Диаметр отложения мокрого снега составляет 5 мм.': 'current_weather_Non-showery snow Diameter of wet snow deposit is 5 mm',
    'current_weather_Снег неливневый.  Диаметр отложения мокрого снега составляет 6 мм.': 'current_weather_Non-showery snow Diameter of ice deposit is 1 mm',
    'current_weather_Снег неливневый.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Non-showery snow Diameter of ice deposit is 2 mm',
    'current_weather_Снег неливневый.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Non-showery snow Diameter of ice deposit is 4 mm',
    'current_weather_Снег неливневый.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Non-showery snow Diameter of mixed deposit is 1 mm',
    'current_weather_Снег неливневый.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Non-showery snow Diameter of mixed deposit is 1 mm',
    'current_weather_Снег непрерывный сильный в срок наблюдения. ': 'current_weather_Continuous heavy snowfall within the observation period',
    'current_weather_Снег непрерывный слабый в срок наблюдения. ': 'current_weather_Continuous light snowfall within the observation period',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр изморозевого отложения составляет 1 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 1 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 2 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 3 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр изморозевого отложения составляет 4 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 4 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр изморозевого отложения составляет 6 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 6 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр изморозевого отложения составляет 7 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 7 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 1 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of wet snow deposit is 1 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of wet snow deposit is 2 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 3 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of wet snow deposit is 3 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 4 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of wet snow deposit is 4 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 5 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of wet snow deposit is 5 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения мокрого снега составляет 7 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of wet snow deposit is 7 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 1 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 2 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения при гололеде составляет 3 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 3 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of ice deposit is 4 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of mixed deposit is 1 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр смешанного отложения составляет 2 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of mixed deposit is 2 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр смешанного отложения составляет 3 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of mixed deposit is 3 mm',
    'current_weather_Снег непрерывный слабый в срок наблюдения.  Диаметр смешанного отложения составляет 4 мм.': 'current_weather_Continuous light snowfall within the observation period Diameter of mixed deposit is 4 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения. ': 'current_weather_Continuous moderate snowfall within the observation period',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения мокрого снега составляет 1 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of wet snow deposit is 1 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of wet snow deposit is 2 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения мокрого снега составляет 3 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of wet snow deposit is 3 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения мокрого снега составляет 4 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of wet snow deposit is 4 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения мокрого снега составляет 8 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of wet snow deposit is 8 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of ice deposit is 1 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of ice deposit is 2 mm',
    'current_weather_Снег непрерывный умеренный в срок наблюдения.  Диаметр смешанного отложения составляет 4 мм.': 'current_weather_Continuous moderate snowfall within the observation period Diameter of mixed deposit is 4 mm',
    'current_weather_Снег с перерывами сильный в срок наблюдения. ': 'current_weather_Intermittent heavy snowfall within the observation period',
    'current_weather_Снег с перерывами сильный в срок наблюдения.  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_Intermittent heavy snowfall within the observation period Diameter of ice deposit is 3 mm',
    'current_weather_Снег с перерывами слабый в срок наблюдения. ': 'current_weather_Intermittent light snowfall within the observation period',
    'current_weather_Снег с перерывами умеренный в срок наблюдения. ': 'current_weather_Intermittent moderate snowfall within the observation period',
    'current_weather_Снежные зерна (с туманом или без него). ': 'current_weather_Snow grains with or without fog',
    'current_weather_Состояние неба в общем не изменилось. ': 'current_weather_Sky condition generally unchanged',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр изморозевого отложения составляет 1 мм.': 'current_weather_Sky condition generally unchanged Diameter of ice deposit is 1 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_Sky condition generally unchanged Diameter of ice deposit is 2 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_Sky condition generally unchanged Diameter of ice deposit is 3 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения мокрого снега составляет 1 мм.': 'current_weather_Sky condition generally unchanged Diameter of wet snow deposit is 1 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения мокрого снега составляет 2 мм.': 'current_weather_Sky condition generally unchanged Diameter of wet snow deposit is 2 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения мокрого снега составляет 3 мм.': 'current_weather_Sky condition generally unchanged Diameter of wet snow deposit is 3 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения мокрого снега составляет 4 мм.': 'current_weather_Sky condition generally unchanged Diameter of wet snow deposit is 4 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Sky condition generally unchanged Diameter of ice deposit is 1 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Sky condition generally unchanged Diameter of ice deposit is 2 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр отложения при гололеде составляет 4 мм.': 'current_weather_Sky condition generally unchanged Diameter of ice deposit is 4 mm',
    'current_weather_Состояние неба в общем не изменилось.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Sky condition generally unchanged Diameter of mixed deposit is 1 mm',
    'current_weather_Состояние неба в общем не изменилось.  Максимальный диаметр градин составляет 2 мм.': 'current_weather_Sky condition generally unchanged Maximum diameter of hailstones is 2 mm',
    'current_weather_Состояние неба в общем не изменилось.  Максимальный диаметр градин составляет 3 мм.': 'current_weather_Sky condition generally unchanged Maximum diameter of hailstones is 3 mm',
    'current_weather_Состояние неба в общем не изменилось.  Максимальный диаметр градин составляет 4 мм.': 'current_weather_Sky condition generally unchanged Maximum diameter of hailstones is 4 mm',
    'current_weather_Туман или ледяной туман, неба не видно, без заметного изменения интенсивности в течение последнего часа. ': 'current_weather_Fog or freezing fog sky obscured without significant change in intensity during the last hour',
    'current_weather_Туман или ледяной туман, неба не видно, без заметного изменения интенсивности в течение последнего часа.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Fog or freezing fog sky obscured without significant change in intensity during the last hour Diameter of mixed deposit is 1 mm',
    'current_weather_Туман или ледяной туман, неба не видно, начался или усилился в течение последнего часа. ': 'current_weather_Fog or freezing fog sky obscured started or intensified during the last hour',
    'current_weather_Туман или ледяной туман, неба не видно, начался или усилился в течение последнего часа.  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_Fog or freezing fog sky obscured started or intensified during the last hour Diameter of ice deposit is 2 mm',
    'current_weather_Туман или ледяной туман, неба не видно, начался или усилился в течение последнего часа.  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_Fog or freezing fog sky obscured started or intensified during the last hour Diameter of ice deposit is 3 mm',
    'current_weather_Туман или ледяной туман, неба не видно, начался или усилился в течение последнего часа.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Fog or freezing fog sky obscured started or intensified during the last hour Diameter of ice deposit is 1 mm',
    'current_weather_Туман или ледяной туман, неба не видно, ослабел за последний час. ': 'current_weather_Fog or freezing fog sky obscured weakened during the last hour',
    'current_weather_Туман или ледяной туман, неба не видно, ослабел за последний час.  Диаметр изморозевого отложения составляет 4 мм.': 'current_weather_Fog or freezing fog sky obscured weakened during the last hour Diameter of ice deposit is 4 mm',
    'current_weather_Туман или ледяной туман, небо видно, без заметного изменения интенсивности в течение последнего часа. ': 'current_weather_Fog or freezing fog sky visible without significant change in intensity during the last hour',
    'current_weather_Туман или ледяной туман, небо видно, начался или усилился в течение последнего часа. ': 'current_weather_Fog or freezing fog sky visible started or intensified during the last hour',
    'current_weather_Туман или ледяной туман, небо видно, ослабел за последний час. ': 'current_weather_Fog or freezing fog sky visible weakened during the last hour',
    'current_weather_Туман или ледяной туман. ': 'current_weather_Fog or freezing fog',
    'current_weather_Туман или ледяной туман.  Диаметр изморозевого отложения составляет 1 мм.': 'current_weather_Fog or freezing fog Diameter of ice deposit is 1 mm',
    'current_weather_Туман или ледяной туман.  Диаметр изморозевого отложения составляет 2 мм.': 'current_weather_Fog or freezing fog Diameter of ice deposit is 2 mm',
    'current_weather_Туман или ледяной туман.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Fog or freezing fog Diameter of ice deposit is 1 mm',
    'current_weather_Туман или обрывки ледяного тумана. ': 'current_weather_Fog or patches of freezing fog',
    'current_weather_Туман с отложением изморози, неба не видно. ': 'current_weather_Fog with freezing fog deposit sky obscured',
    'current_weather_Туман с отложением изморози, неба не видно.  Диаметр изморозевого отложения составляет 3 мм.': 'current_weather_Fog with freezing fog deposit sky obscured Diameter of ice deposit is 3 mm',
    'current_weather_Туман с отложением изморози, неба не видно.  Диаметр отложения при гололеде составляет 1 мм.': 'current_weather_Fog with freezing fog deposit sky obscured Diameter of ice deposit is 1 mm',
    'current_weather_Туман с отложением изморози, неба не видно.  Диаметр отложения при гололеде составляет 2 мм.': 'current_weather_Fog with freezing fog deposit sky obscured Diameter of ice deposit is 2 mm',
    'current_weather_Туман с отложением изморози, неба не видно.  Диаметр смешанного отложения составляет 1 мм.': 'current_weather_Fog with freezing fog deposit sky obscured Diameter of mixed deposit is 1 mm',
    'current_weather_Туман с отложением изморози, небо видно. ': 'current_weather_Fog with freezing fog deposit sky visible',
    'current_weather_Умеренный или сильный дождь в срок наблюдения. Гроза в течение последнего часа, но не в срок наблюдения. ': 'current_weather_Moderate or heavy rain within the observation period Thunderstorm occurred within the last hour but not during the observation period',
    'weather_after_1time_Буря': 'weather_after_1time_Storm',
    'weather_after_1time_Гроза (грозы) с осадками или без них.': 'weather_after_1time_Thunderstorm thunderstorms with or without precipitation',
    'weather_after_1time_Дождь со снегом или другими видами твердых осадков': 'weather_after_1time_Rain with snow or other types of solid precipitation',
    'weather_after_1time_Дождь.': 'weather_after_1time_Rain',
    'weather_after_1time_Ливень (ливни).': 'weather_after_1time_Downpour downpours',
    'weather_after_1time_Метель': 'weather_after_1time_Blizzard',
    'weather_after_1time_Морось.': 'weather_after_1time_Drizzle',
    'weather_after_1time_Облака покрывали более половины неба в течение всего соответствующего периода.': 'weather_after_1time_More than half of the sky covered by clouds throughout the corresponding period',
    'weather_after_1time_Облака покрывали более половины неба в течение одной части соответствующего периода и половину или менее в течение другой части периода.': 'weather_after_1time_More than half of the sky covered by clouds during one part of the corresponding period and half or less during another part of the period',
    'weather_after_1time_Облака покрывали половину неба или менее в течение всего соответствующего периода.': 'weather_after_1time_Half of the sky covered by clouds or less throughout the corresponding period',
    'weather_after_1time_Снег и/или другие виды твердых осадков': 'weather_after_1time_Snow and/or other types of solid precipitation',
    'weather_after_1time_Туман или ледяной туман или сильная мгла.': 'weather_after_1time_Fog or freezing fog or thick fog',
    'weather_after_2time_Буря': 'weather_after_2time_Storm',
    'weather_after_2time_Дождь со снегом или другими видами твердых осадков': 'weather_after_2time_Rain with snow or other types of solid precipitation',
    'weather_after_2time_Дождь.': 'weather_after_2time_Rain',
    'weather_after_2time_Ливень (ливни).': 'weather_after_2time_Downpour downpours',
    'weather_after_2time_Метель': 'weather_after_2time_Blizzard',
    'weather_after_2time_Морось.': 'weather_after_2time_Drizzle',
    'weather_after_2time_Облака покрывали более половины неба в течение всего соответствующего периода.': 'weather_after_2time_More than half of the sky covered by clouds throughout the corresponding period',
    'weather_after_2time_Облака покрывали более половины неба в течение одной части соответствующего периода и половину или менее в течение другой части периода.': 'weather_after_2time_More than half of the sky covered by clouds during one part of the corresponding period and half or less during another part of the period',
    'weather_after_2time_Облака покрывали половину неба или менее в течение всего соответствующего периода.': 'weather_after_2time_Half of the sky covered by clouds or less throughout the corresponding period',
    'weather_after_2time_Снег и/или другие виды твердых осадков': 'weather_after_2time_Snow and/or other types of solid precipitation',
    'weather_after_2time_Туман или ледяной туман или сильная мгла.': 'weather_after_2time_Fog or freezing fog or thick fog',
    'clouds_middle_level_0': 'clouds_middle_level_0',
    'clouds_middle_level_Высококучевые башенкообразные или хлопьевидные.': 'clouds_middle_level_High cumuliform towers or floccus',
    'clouds_middle_level_Высококучевые при хаотическом виде неба, обычно расположенные на нескольких уровнях.': 'clouds_middle_level_High cumuliform clouds in a chaotic sky usually at multiple levels',
    'clouds_middle_level_Высококучевые просвечивающие, полосами, либо один или несколько слоев высококучевых просвечивающих, распространяющихся по небу; эти высококучевые в целом уплотняются.': 'clouds_middle_level_High cumuliform transmissive in bands or one or more layers of high cumuliform transmissive clouds spread across the sky these high cumuliform clouds are generally densifying',
    'clouds_middle_level_Высококучевые просвечивающие, расположенные на одном уровне.': 'clouds_middle_level_High cumuliform transmissive at one level',
    'clouds_middle_level_Высококучевые, образовавшиеся из кучевых (или кучево-дождевых).': 'clouds_middle_level_High cumuliform formed from cumulus or cumulonimbus',
    'clouds_middle_level_Высококучевые, просвечивающие или плотные в двух или более слоях или высококучевые плотные в одном слое, не распространяющиеся но небу, либо высококучевые с высокослоистыми или слоисто-дождевыми.': 'clouds_middle_level_High cumuliform transmissive or dense in two or more layers or high cumuliform dense in one layer not extending over the sky or high cumuliform with high stratiform or stratiform-rain',
    'clouds_middle_level_Высококучевых, высокослоистых или слоисто-дождевых облаков нет.': 'clouds_middle_level_No high cumuliform stratiform or stratiform-rain clouds',
    'clouds_middle_level_Высокослоистые непросвечивающие или слоисто-дождевые.': 'clouds_middle_level_High stratiform non-transmissive or stratiform-rain',
    'clouds_middle_level_Высокослоистые просвечивающие.': 'clouds_middle_level_High stratiform transmissive',
    'clouds_middle_level_Клочья (часто чечевицеобразные) высококучевых просвечивающих, непрерывно изменяющихся; расположены на одном или более уровнях.': 'clouds_middle_level_Fragments often lentil-shaped of high cumuliform transmissive continuously changing located at one or more levels',
    'clouds_high_level_0': 'clouds_high_level_0',
    'clouds_high_level_Перисто-кучевые одни или перисто-кучевые, сопровождаемые перистыми или перисто-слоистыми, либо те и другие, но перисто-кучевые преобладают среди них.': 'clouds_high_level_Cirrocumulus alone or cirrocumulus accompanied by cirrus or cirrostratus or both but cirrocumulus predominates among them',
    'clouds_high_level_Перисто-слоистые, не распространяющиеся по небу и не покрывающие его полностью.': 'clouds_high_level_Cirrostratus not spreading across the sky and not covering it completely',
    'clouds_high_level_Перисто-слоистые, покрывающие все небо.': 'clouds_high_level_Cirrostratus covering the entire sky',
    'clouds_high_level_Перистые (часто в виде полос) и перисто-слоистые, распространяющиеся по небу и в целом обычно уплотняющиеся, но сплошная пелена поднимается над горизонтом менее чем на 45°.': 'clouds_high_level_Cirrus often in the form of bands and cirrostratus spreading across the sky and generally densifying but a solid veil rising above the horizon less than 45',
    'clouds_high_level_Перистые (часто в виде полос) и перисто-слоистые, распространяющиеся по небу и в целом обычно уплотняющиеся; сплошная пелена, поднимающаяся над горизонтом выше 45°, не покрывает всего неба.': 'clouds_high_level_Cirrus often in the form of bands and cirrostratus spreading across the sky and generally densifying a solid veil rising above the horizon higher than 45 not covering the entire sky',
    'clouds_high_level_Перистые когтевидные или нитевидные или первые и вторые, распространяющиеся по небу и в целом обычно уплотняющиеся.': 'clouds_high_level_Cirrus fibratus or uncinus or both spreading across the sky and generally densifying',
    'clouds_high_level_Перистые нитевидные, иногда когтевидные, не распространяющиеся по небу.': 'clouds_high_level_Cirrus uncinus sometimes fibratus not spreading across the sky',
    'clouds_high_level_Перистые плотные в виде клочьев или скрученных склонов, количество которых обычно не увеличивается, иногда могут казаться остатками верхней части кучево-дождевых; или перистые башенкообразные, или перистые хлопьевидные.': 'clouds_high_level_Cirrus dense in patches or twisted slopes the number of which usually does not increase sometimes may appear as remnants of the upper part of cumulonimbus or cirrus towered or cirrus floccus',
    'clouds_high_level_Перистые плотные, образовавшиеся от кучево-дождевых.': 'clouds_high_level_Cirrus dense formed from cumulonimbus',
    'clouds_high_level_Перистых, перисто-кучевых или перисто-слоистых нет.':'clouds_high_level_No cirrus cirrocumulus or cirrostratus',
    'clouds_low_level_Кучево-дождевые волокнистые (часто с наковальней), либо с кучево-дождевыми лысыми, кучевыми, слоистыми, разорванно-дождевыми, либо без них.': 'clouds_low_level_Stratocumulus fibratus often with a roll or with stratocumulus castellanus stratocumulus stratiform fractostratus or without them',
    'clouds_low_level_Кучево-дождевые лысые с кучевыми, слоисто-кучевыми или слоистыми, либо без них.': 'clouds_low_level_Stratocumulus castellanus with stratocumulus stratiform or without them',
    'clouds_low_level_Кучевые и слоисто-кучевые (но не слоисто-кучевые, образовавшиеся из кучевых), основания расположены на разных уровнях.': 'clouds_low_level_Cumulus and stratocumulus but not cumulostratus formed from cumulus bases located at different levels',
    'clouds_low_level_Кучевые плоские или кучевые разорванные, или те и другие вместе, не относящиеся к облакам плохой погоды.': 'clouds_low_level_Cumulus flat or cumulus fractus or both together not associated with bad weather clouds',
    'clouds_low_level_Кучевые средние или мощные или вместе с кучевыми разорванными, или с кучевыми плоскими, или со слоисто-кучевыми, либо без них; основания всех этих облаков расположены на одном уровне.': 'clouds_low_level_Cumulus mediocris or congestus or together with cumulus fractus or with cumulus flat or with stratocumulus or without them bases of all these clouds are at the same level',
    'clouds_low_level_Слоисто-кучевые, образовавшиеся из кучевых.': 'clouds_low_level_Cumulostratus formed from cumulus',
    'clouds_low_level_Слоисто-кучевые, образовавшиеся не из кучевых.': 'clouds_low_level_Cumulostratus not formed from cumulus',
    'clouds_low_level_Слоисто-кучевых, слоистых, кучевых или кучево-дождевых облаков нет.': 'clouds_low_level_No cumulostratus stratocumulus cumulus or cumulonimbus clouds',
    'clouds_low_level_Слоистые разорванные или кучевые разорванные облака плохой погоды, либо те и другие вместе (разорванно-дождевые); обычно расположены под слоистыми или слоисто-дождевыми облаками.': 'clouds_low_level_Stratus fractus or cumulus fractus bad weather clouds or both together fractostratus usually located beneath stratiform or nimbostratus clouds',
    'clouds_low_level_Слоистые туманообразные или слоистые разорванные, либо те и другие, но не относящиеся к облакам плохой погоды.': 'clouds_low_level_Stratus nebulosus or stratocumulus fractus or both but not bad weather clouds'
                    }

# Заменяем названия столбцов с помощью словаря перевода
df = df.rename(columns=translation_dict)

LightGBM

In [19]:
x = train_df_wo_temp
y = train_df[['air_temperature']]
x_test = test_df_wo_temp
y_test = test_df[['air_temperature']]
x_valid = val_df_wo_temp
y_valid = val_df[['air_temperature']]

train_data = lgb.Dataset(x, label=y)
val_data = lgb.Dataset(x_valid, label=y_valid)
test_data = lgb.Dataset(x_test, label=y_test)

params = {
    'max_depth': -1,
    'learning_rate': 0.1,
    'n_estimators': 10_000,
    'random_state': 0,  # for reproducibility
    'min_child_samples': 1,
    # 'min_split_gain': 0,
    # 'reg_alpha': 0,  # no L1 reg
    # 'reg_lambda': 0,  # no L2 reg
    'num_leaves': 25,
    'verbose': 1,
    
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
}

gbm = lgb.train(params, train_data, num_boost_round=100, valid_sets=[val_data])
y_pred = abs(gbm.predict(x_test))
print(ndcg_score, y_pred)

# намеренно увеличиваю на 30 значения предсказания для наглядного запараллеливания графиков
test_prediction_df['test_prediction'] += 30

plt.figure(figsize=(180, 40))  # установка ширины и высоты графика

# Отрисовка графика
plt.plot(test_prediction_df['id'], test_prediction_df['air_temperature'], label='air_temperature', color='red')
plt.plot(test_prediction_df['id'], test_prediction_df['test_prediction'], label='test_prediction', color='blue')

# Добавляем подписи к осям и легенду
plt.xlabel('id')
plt.ylabel('Value')
plt.legend()

# Показываем график
plt.show()

mae = MAE(test_df['air_temperature'], y_pred)
mape = MAPE(test_df['air_temperature'], y_pred)
print("test MAE: ", mae)
print("test MAPE: ", mape)


c:\Users\korde\anaconda3\Lib\site-packages\lightgbm\engine.py:172: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


LightGBMError: Do not support special JSON characters in feature name.

In [47]:
# is cuda available
torch.cuda.is_available()

True

In [55]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        
        self.fc1 = nn.Linear(291, 1_000)
        self.fc2 = nn.Linear(1_000, 10_000)
        self.fc3 = nn.Linear(10_000, 1_000)
        self.fc4 = nn.Linear(1_000, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        
        return x

In [49]:
scaler = StandardScaler()

numerical_columns = train_df_wo_temp.select_dtypes(include=['float64', 'int64']).columns

x_train_normalized = train_df_wo_temp.copy()  
x_train_normalized[numerical_columns] = scaler.fit_transform(x_train_normalized[numerical_columns])

numerical_columns = test_df_wo_temp.select_dtypes(include=['float64', 'int64']).columns

x_test_normalized = test_df_wo_temp.copy()  
x_test_normalized[numerical_columns] = scaler.fit_transform(x_test_normalized[numerical_columns])

numerical_columns = val_df_wo_temp.select_dtypes(include=['float64', 'int64']).columns
x_val_normalized = val_df_wo_temp.copy()  
x_val_normalized[numerical_columns] = scaler.fit_transform(x_val_normalized[numerical_columns])

In [50]:
x_train_normalized = x_train_normalized.astype('float32')
X_tensor = torch.tensor(x_train_normalized.values, dtype=torch.float32)

y_tensor = torch.tensor(train_df['air_temperature'].values, dtype=torch.float32).view(-1, 1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

X_tensor, y_tensor = X_tensor.to(device), y_tensor.to(device)
print(X_tensor.size())
print(y_tensor.size())

Using device: cuda
torch.Size([38872, 291])
torch.Size([38872, 1])


In [51]:
class RMSELoss(nn.Module):
    def __init__(self):
        super(RMSELoss, self).__init__()

    def forward(self, y_pred, y_true):
        mse = nn.MSELoss()(y_pred, y_true)
        rmse = torch.sqrt(mse)
        return rmse


In [56]:
from tqdm import tqdm

net = Net().to(device)
criterion = RMSELoss()
optimizer = optim.Adam(net.parameters(), lr=0.01)

n_epoch = 15_000
loss_values = []
intermediate_predictions = {}

pbar = tqdm(range(n_epoch))

for epoch in pbar:
    # nullify gradients from the previous iterations
    optimizer.zero_grad()
    # make prediction
    outputs = net(X_tensor)
    # calculate loss
    loss = criterion(outputs, y_tensor)
    # make backpropagation, calculate gradients
    loss.backward()
    # make gradient descent step
    optimizer.step()
    
    # save loss
    loss_values.append(loss.item())
    
    # save intermediate predictions
    intermediate_predictions[epoch] = net(X_tensor).detach().cpu().numpy()
    
    # output progress
    if epoch %10 == 0:
        pbar.set_description(f"Epoch {epoch}/{n_epoch} | Loss: {loss_values[-1]:.4f}")

plt.plot(loss_values)
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

Epoch 2850/15000 | Loss: 0.2841:  19%|█▉        | 2859/15000 [1:04:18<4:33:04,  1.35s/it]


KeyboardInterrupt: 